Validasi GeoJSON

In [9]:
import geopandas as gpd
from shapely.geometry import box

In [10]:
PATH = "./digitizations/training_samples.geojson"
BBOX = (113.25, -8.84, 114.64, -7.56)

In [11]:
gdf = gpd.read_file(PATH)
gdf = gdf.set_crs(4326) if gdf.crs is None else gdf.to_crs(4326)
m = gdf.to_crs(32749)
gdf['area_m2'] = m.geometry.area

In [12]:
print(gdf.groupby('class').agg(n=('class', 'size'),
                               luas_median_m2=('area_m2', 'median'),
                               luas_min_m2=('area_m2', 'min'),
                               luas_total_ha=('area_m2', lambda s: s.sum() / 1e4)).round(0))
print('Geometri tidak valid  :', int((~gdf.is_valid).sum()))
print('Poligon < 2500 m2     :', int((gdf.area_m2 < 2500).sum()))
print('Di luar bbox raster   :', int((~m.geometry.centroid.to_crs(4326).within(box(*BBOX))).sum()))

            n  luas_median_m2  luas_min_m2  luas_total_ha
class                                                    
built_up  100          4336.0          0.0          176.0
farmland  100         11126.0       1977.0          294.0
forest    100        520217.0       7855.0        42163.0
mangrove  115          2814.0        215.0          123.0
water      95         72283.0        497.0       209133.0
Geometri tidak valid  : 2
Poligon < 2500 m2     : 95
Di luar bbox raster   : 0


In [13]:
ov = gpd.overlay(m[['class', 'geometry']].reset_index(), m[['class', 'geometry']].reset_index(),
                 how='intersection', keep_geom_type=True)

ov = ov[(ov['index_1'] < ov['index_2']) & (ov['class_1'] != ov['class_2'])]

In [14]:
print('Tumpang tindih beda kelas:', len(ov))

Tumpang tindih beda kelas: 2


Jika ada yang tumpang tindih, jalankan ini untuk mengecek detailnya

In [15]:
import geopandas as gpd
import os

PATH = 'digitizations/training_samples.geojson'
gdf = gpd.read_file(PATH).to_crs(4326)
m = gdf.to_crs(32749).reset_index(drop=True)

# 1) cari pasangan yang tumpang tindih beda kelas
a = m[['class', 'geometry']].reset_index()
ov = gpd.overlay(a, a, how='intersection', keep_geom_type=True)
ov = ov[(ov['index_1'] < ov['index_2']) & (ov['class_1'] != ov['class_2'])].copy()
ov['area_m2'] = ov.geometry.area
ov['pct_poligon_1'] = 100 * ov['area_m2'] / m.geometry.area.loc[ov['index_1']].values
ov['pct_poligon_2'] = 100 * ov['area_m2'] / m.geometry.area.loc[ov['index_2']].values
print(ov[['index_1', 'class_1', 'index_2', 'class_2', 'area_m2', 'pct_poligon_1', 'pct_poligon_2']].round(1))

# 2) bersihkan: potong area tumpang tindih dari KEDUA poligon
for _, r in ov.iterrows():
    cut = r.geometry.buffer(1)
    for k in (r['index_1'], r['index_2']):
        m.loc[k, 'geometry'] = m.loc[k, 'geometry'].difference(cut)

m = m[~m.geometry.is_empty].explode(index_parts=False).reset_index(drop=True)
m = m[m.geometry.area >= 2500]
clean = m.to_crs(4326)
os.makedirs('./digitizations/clean', exist_ok=True)
clean.to_file('./digitizations/clean/training_samples_clean.geojson', driver='GeoJSON')
for c, g in clean.groupby('class'):
    g.to_file(f'./digitizations/clean/training_{c}.geojson', driver='GeoJSON')
print(clean['class'].value_counts())

    index_1 class_1  index_2   class_2  area_m2  pct_poligon_1  pct_poligon_2
74       63   water      267  mangrove    122.3            0.1            0.4
77       65   water      271  mangrove      1.6            0.0            0.0
class
forest      100
farmland     99
water        84
built_up     68
mangrove     64
Name: count, dtype: int64


Jalankan ini untuk membandingkan sebelum dibersihkan dan sesudah dibersihkan

In [16]:
print(gdf['class'].value_counts())      # sebelum
print(clean['class'].value_counts())    # sesudah

class
mangrove    115
farmland    100
forest      100
built_up    100
water        95
Name: count, dtype: int64
class
forest      100
farmland     99
water        84
built_up     68
mangrove     64
Name: count, dtype: int64
